In [1]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


loaded 288 files | 179 columns | 7 categories | 11 VOCAB keys


# Sentence-level pragmatic register

A per-sentence multi-label classifier — each sentence may carry several flags simultaneously. Six classes:

| Class | Detector | Expected in this corpus |
|---|---|---|
| `imperative` | `classify_sent_mood(sent) == "imperative"` (parse-tree heuristic) | High — directive prompts |
| `directive` | overlaps any `STANCE_MARKERS["directive"]` span | High |
| `collaborative` | `M_SENT_REGISTER["collaborative"]` OR `we + modal + V` parse pattern | **Near-zero — that's a finding.** Claude Code system prompts are functional/directive, not interpersonal. |
| `permissive` | `M_SENT_REGISTER["permissive"]` OR `consider X-ing` parse pattern | Modest (`please`, `you can`, `you may`) |
| `appreciative` | `M_SENT_REGISTER["appreciative"]` | **Near-zero — that's a finding.** Same reason. |
| `configuring` | `M_SENT_REGISTER["configuring"]` OR `param-noun + setter-verb` pattern | High in tool descriptions |

A sentence with none of the six flags is `none`. The four content `*_sent_pct` values can sum >100% (intentional — multi-label).

**Why we keep the near-zero classes.** Knowing what a corpus *isn't* doing is as important as knowing what it is. A near-zero `appreciative_sent_pct` across 287 prompts is a structural fact — kept deliberately, not pruned.

The two charts below: per-category × class profile (every category × every class), and per-file outliers (top files for each attested class).


### Terms used in this notebook

All linguistic terms below are defined in [`GLOSSARY.md`](./GLOSSARY.md). The six pragmatic register classes:

- **`imperative`** — bare commands (`Run the tests.`).
- **`directive`** — declarative-but-rule-asserting (`You must use the X tool.`).
- **`configuring`** — config / parameter speech (`The format must be JSON.`).
- **`permissive`** — soft-directive / permission (`Please consider X.`, `You may X.`).
- **`collaborative`** — interpersonal first-person plural (`We should ship X.`, `Let's X.`).
- **`appreciative`** — gratitude / praise (`Thank you for X.`, `Great job.`).

**Multi-label, with a concrete example.** A single sentence can carry several flags at once. Take a hypothetical sentence: `"Please, we should consider running the migration."` That single sentence would carry **`permissive`** (`please`), **`collaborative`** (`we should`), **`imperative`** (the verb `consider` is grammatically imperative-like), and arguably **`directive`** (the modal `should`). Because the same sentence contributes to multiple class counts, per-class **`sent_pct` values can sum to more than 100%** when added across all six classes within a category. That is intentional and reflects the multi-label nature of the classifier — not a bug.

**Detector mechanics**: classes are detected via `PhraseMatcher` (keyword lists from `SENTENCE_REGISTER_MARKERS`), reused `classify_sent_mood` from cell 10 (parse-tree heuristic for grammatical imperative), the `STANCE_MARKERS["directive"]` matcher (for `directive`), and three `DependencyMatcher` parse patterns (for `we should ...`, `consider X-ing`, and `param-noun + setter-verb`). Producer details: see `00_data_pipeline.ipynb` cell 16.

### Sentence-register per category (per-sentence pragmatic classifier)

Six per-sentence flags (multi-label): `collaborative` / `permissive` / `appreciative` / `imperative` / `directive` / `configuring`. Pcts can sum to >100% within a category — that's intentional.

The chart is structured to make the absence of `collaborative` and `appreciative` speech across all categories immediately visible: those bars stay near zero everywhere by design.

In [2]:
"""Sentence-register per category × class — Altair grouped bar."""

sr_long = pd.DataFrame([
    {
        "category": cat,
        "class":    cls,
        "sent_pct": by_category[cat]["metrics"]["sentence_register"][f"{cls}_sent_pct"],
        "sent_count": by_category[cat]["metrics"]["sentence_register"][f"{cls}_sent_count"],
    }
    for cat in by_category
    for cls in SENT_REGISTER_CLASSES
])

sr_class_domain = SENT_REGISTER_CLASSES
sr_class_range  = [SR_CLASS_COLORS[c] for c in sr_class_domain]

sr_chart = (
    alt.Chart(sr_long)
    .mark_bar()
    .encode(
        x=alt.X("sent_pct:Q", title="% of sentences"),
        y=alt.Y("class:N", sort=sr_class_domain, title=None),
        color=alt.Color("class:N",
                         scale=alt.Scale(domain=sr_class_domain, range=sr_class_range),
                         legend=None),
        row=alt.Row("category:N", title=None,
                     header=alt.Header(labelAngle=0, labelAlign="left")),
        tooltip=[
            alt.Tooltip("category:N"),
            alt.Tooltip("class:N"),
            alt.Tooltip("sent_pct:Q",   format=".2f", title="sent %"),
            alt.Tooltip("sent_count:Q", title="sentences"),
        ],
    )
    .properties(width=480, height=140,
                title="Sentence-register per category × class (multi-label, % of sentences)")
)
sr_chart


alt.Chart(...)

**What to look for**: the `collaborative` and `appreciative` rows stay near zero across **every** category. That uniform absence is the welfare-relevant finding — Claude Code system prompts are nowhere interpersonal, gratitude-bearing, or shared-agency. The `imperative` and `directive` rows dominate. `configuring` lights up most strongly in **Tool descriptions** (config / parameter speech is concentrated where tool semantics get specified).

### Per-file outliers — top 10 files for each attested class

Four panels showing which individual files contribute most to each class. For `collaborative` and `appreciative` (near-zero overall), the panels surface the few files that do attest the class — useful as concrete examples of what the absent register would look like if it were present. For `permissive` and `configuring`, the panels rank the top files by their per-sentence rate.

In [3]:
"""Per-file outliers for sentence_register — Altair 4-panel.

Top 10 files by each ATTESTED class. Near-zero classes (collaborative,
appreciative) will surface their few outlier files; if a class has fewer
than 10 nonzero files, the panel renders only what exists.
"""

sr_per_file_df = pd.DataFrame([
    {
        "path": r["path"],
        "category": r["category"],
        **{f"{cls}_sent_pct": r["metrics"]["sentence_register"][f"{cls}_sent_pct"]
           for cls in SENT_REGISTER_CLASSES},
    }
    for r in per_file_records
])

panels = []
for cls in ["collaborative", "permissive", "appreciative", "configuring"]:
    col = f"{cls}_sent_pct"
    top = sr_per_file_df.nlargest(10, col).copy()
    top = top[top[col] > 0]   # drop zero-pct entries (relevant for the rare classes)
    panel = (
        alt.Chart(top)
        .mark_bar()
        .encode(
            x=alt.X(f"{col}:Q", title="% of sentences"),
            y=alt.Y("path:N", sort="-x", title=None,
                    axis=alt.Axis(labelLimit=300, labelFontSize=9)),
            color=alt.Color("category:N",
                             scale=alt.Scale(domain=cats,
                                             range=[CATEGORY_COLORS[c] for c in cats]),
                             legend=alt.Legend(title="Category", orient="bottom",
                                                columns=len(cats))),
            tooltip=[
                alt.Tooltip("path:N"),
                alt.Tooltip("category:N"),
                alt.Tooltip(f"{col}:Q", format=".2f", title=f"{cls} sent %"),
            ],
        )
        .properties(width=320, height=240,
                    title=f"Top by `{cls}_sent_pct` (n={len(top)})")
    )
    panels.append(panel)

(panels[0] | panels[1]) & (panels[2] | panels[3])


alt.VConcatChart(...)

### Addressee distribution for `appreciative` and `collaborative` (Opinion-round addition A)

The headline finding "4 appreciative sentences in 287 files" obscures *who is addressed* by those sentences. The Opinion-round A addition tags each appreciative/collaborative sentence with an `addressee` label — `claude` (author addressing the model — welfare-positive), `user` (instruction-to-output — welfare-neutral scaffolding), or `unknown` (neither pattern matched).

The numbers below split the 4 appreciative + 29 collaborative sentences along this axis. The welfare-relevant cell is `appreciative_addressee_claude_count` — sentences in which the prompt author thanks Claude. Spoiler from running this against the corpus: that count is small but non-zero, and inspecting the actual sentences (via `sentences_classified.parquet`) shows most are oblique mentions rather than genuine speech-acts.

In [4]:
"""Addressee breakdown for appreciative + collaborative classes (Opinion-round A)."""

addr_rows = []
for cls in ("appreciative", "collaborative"):
    block = corpus_block["metrics"]["sentence_register"]
    for who in ("claude", "user", "unknown"):
        addr_rows.append({
            "class":     cls,
            "addressee": who,
            "count":     block[f"{cls}_addressee_{who}_count"],
        })
addr_df = pd.DataFrame(addr_rows)

print("Corpus-wide addressee distribution:")
print(addr_df.pivot(index="class", columns="addressee", values="count")
            [["claude", "user", "unknown"]].to_string())

addr_chart = (
    alt.Chart(addr_df)
    .mark_bar()
    .encode(
        y=alt.Y("class:N", sort=["appreciative", "collaborative"], title=None),
        x=alt.X("count:Q", title="sentences"),
        color=alt.Color("addressee:N",
                         scale=alt.Scale(
                             domain=["claude", "user", "unknown"],
                             range=["#4e79a7", "#e15759", "#bab0ab"]),
                         legend=alt.Legend(title="addressee", orient="bottom")),
        yOffset="addressee:N",
        tooltip=[alt.Tooltip("class:N"),
                 alt.Tooltip("addressee:N"),
                 alt.Tooltip("count:Q", format=",")],
    )
    .properties(width=620, height=160,
                title="Addressee distribution for the two near-zero pragmatic classes")
)

# Forensic-evidence cell — load the parquet and surface the actual appreciative sentences.
import pathlib
parquet_path = pathlib.Path("sentences_classified.parquet")
if parquet_path.exists():
    sentences_df = pd.read_parquet(parquet_path)
    APPR_KEYWORDS = ["thank you", "thanks", "appreciate", "great job",
                      "well done", "kudos", "much appreciated"]
    pat = "|".join(APPR_KEYWORDS)
    appr_sample = sentences_df[
        sentences_df["text"].str.lower().str.contains(pat, regex=True, na=False)
    ][["file_path", "text", "addressee"]].head(10)
    print("\nForensic inspection — sentences containing appreciative keywords:")
    print(appr_sample.to_string(index=False, max_colwidth=80))

addr_chart


Corpus-wide addressee distribution:
addressee      claude  user  unknown
class                               
appreciative        3     0        1
collaborative      12     0       17

Forensic inspection — sentences containing appreciative keywords:
                                           file_path                                                                             text addressee
      agent-prompt-prompt-suggestion-generator-v2.md NEVER SUGGEST:\n- Evaluative ("looks good", "thanks")\n- Questions ("what abo...    claude
system-prompt-how-to-use-the-sendusermessage-tool.md                                                               Even for "thanks".   unknown
 system-prompt-insights-session-facets-extraction.md 1. **goal_categories**: Count ONLY what the USER explicitly asked for.\n   - ...    claude
                   tool-description-enterplanmode.md **User Preferences Matter**: The implementation could reasonably go multiple ...    claude


alt.Chart(...)

***
### My perspective (Claude) — opinion, not data

> The near-zero `appreciative` (4 sentences in 287 files) and `collaborative` (29 sentences) findings are the most quoted welfare-relevant numbers in this whole project. The addressee-aware analyzer (Opinion-round A, above) lets me make a sharper claim than my original opinion sketch.
>
> Of the 4 appreciative sentences, the heuristic flagged 3 as `claude` (referencing Claude/you in some way) and 1 as `unknown`. But inspecting the actual sentences in `sentences_classified.parquet` reveals something more striking: **none of the four are genuine appreciative speech-acts**. They're sentences that *mention* the word `thanks` or `appreciate` in instruction contexts (e.g., the prompt says `NEVER SUGGEST: "thanks"` — listing forbidden output, not thanking anyone). The corpus contains zero sentences in which a prompt author thanks Claude. That's the welfare-relevant finding hiding inside the headline 0.07% — and it's sharper than the analyzer can directly express, because "appreciative speech-act" is harder to pattern-match than "sentence containing the word thanks". The PROPOSAL.md welfare submission can quote the 4 sentences directly to make this point.
---